[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/produccion/03-optimizacion-costos.ipynb)

# Optimización de Costos en LLMs

> Aprende las técnicas más efectivas para reducir el coste de tus aplicaciones LLM:
> conteo preciso de tokens, Prompt Caching de Anthropic y estrategias de ahorro.

## Objetivos
- Contar tokens con la API de Anthropic antes de hacer llamadas
- Demostrar Prompt Caching: mismo bloque de contexto 3 veces
- Medir el ahorro real con `cache_read_input_tokens`
- Calcular y visualizar el ahorro acumulado

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas matplotlib --quiet

## 2. Configuración y precios

In [ ]:
import anthropic
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Precios oficiales claude-haiku-4-5 (USD por millón de tokens)
PRECIOS = {
    "input_normal":   0.80,   # tokens de entrada normales
    "cache_write":    1.00,   # escribir al caché (25% más que input normal)
    "cache_read":     0.08,   # leer del caché (90% más barato que input normal)
    "output":         4.00    # tokens de salida
}

def calcular_coste(tokens_entrada: int, tokens_salida: int,
                   cache_write: int = 0, cache_read: int = 0) -> float:
    """Calcula el coste en USD de una llamada."""
    return (
        tokens_entrada  / 1_000_000 * PRECIOS["input_normal"] +
        cache_write     / 1_000_000 * PRECIOS["cache_write"] +
        cache_read      / 1_000_000 * PRECIOS["cache_read"] +
        tokens_salida   / 1_000_000 * PRECIOS["output"]
    )

print("Precios por millón de tokens:")
for k, v in PRECIOS.items():
    print(f"  {k:20s}: ${v:.2f}")

## 3. Contador de tokens con la API de Anthropic

Antes de hacer una llamada costosa, podemos contar los tokens exactos que consumirá.

In [ ]:
# Texto largo que simula un documento de contexto (manual de producto, base de conocimiento, etc.)
CONTEXTO_LARGO = """
MANUAL DE PRODUCTO: Sistema de Gestión de Inventario v3.0

CAPÍTULO 1 - INTRODUCCIÓN
El Sistema de Gestión de Inventario (SGI) es una solución empresarial completa para el control
y seguimiento de activos, productos y materiales. Diseñado para empresas de cualquier tamaño,
el SGI ofrece trazabilidad completa desde la recepción hasta el despacho.

CAPÍTULO 2 - MÓDULOS PRINCIPALES
2.1 Módulo de Recepción: Registra entradas de mercancía con código QR/barras, fecha, proveedor
y lote. Soporte para importación masiva desde CSV/Excel. Alertas automáticas por discrepancias.

2.2 Módulo de Almacenamiento: Gestión de ubicaciones (pasillo, estante, nivel). Mapa visual
del almacén. Rotación FIFO/LIFO configurable. Control de condiciones (temperatura, humedad).

2.3 Módulo de Despacho: Órdenes de picking optimizadas. Integración con sistemas de transporte.
Documentación automática: albaranes, facturas, certificados de calidad.

2.4 Módulo de Reportes: Dashboard en tiempo real. Reportes de rotación, merma, valoración
de inventario (FIFO, LIFO, promedio ponderado). Exportación a PDF/Excel/CSV.

CAPÍTULO 3 - INTEGRACIÓN CON SISTEMAS EXTERNOS
El SGI se integra mediante API REST (OpenAPI 3.0) con: ERP SAP, Oracle, Microsoft Dynamics;
plataformas de e-commerce Shopify, WooCommerce, Magento; y sistemas contables.

CAPÍTULO 4 - SEGURIDAD Y AUDITORÍA
Control de acceso basado en roles (RBAC). Registro de auditoría inmutable. Cifrado AES-256
en reposo y TLS 1.3 en tránsito. Cumplimiento ISO 27001 y GDPR.
"""

mensajes_ejemplo = [
    {"role": "user", "content": CONTEXTO_LARGO + "\n\n¿Qué módulos tiene el sistema?"}
]

# Contar tokens ANTES de hacer la llamada
conteo = client.messages.count_tokens(
    model=MODELO,
    messages=mensajes_ejemplo
)

tokens_entrada = conteo.input_tokens
tokens_salida_estimados = 200  # estimación para el cálculo previo
coste_estimado = calcular_coste(tokens_entrada, tokens_salida_estimados)

print(f"Tokens de entrada: {tokens_entrada:,}")
print(f"Coste estimado (con ~{tokens_salida_estimados} tokens de salida): ${coste_estimado:.6f} USD")
print(f"Coste por 1,000 llamadas iguales: ${coste_estimado * 1000:.4f} USD")

## 4. Demostración de Prompt Caching

Enviamos el mismo bloque de contexto 3 veces con `cache_control`.
La primera llamada escribe al caché; las siguientes lo leen a 1/10 del precio.

In [ ]:
preguntas = [
    "¿Cuáles son los módulos principales del sistema?",
    "¿Qué sistemas externos se pueden integrar con el SGI?",
    "¿Cómo garantiza la seguridad el sistema?"
]

resultados_cache = []

print("Realizando 3 llamadas con el mismo contexto...")
print("-" * 80)

for i, pregunta in enumerate(preguntas, 1):
    # El bloque de contexto grande lleva cache_control para activar el caché
    mensajes = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": CONTEXTO_LARGO,
                    "cache_control": {"type": "ephemeral"}  # <- activa el caché
                },
                {
                    "type": "text",
                    "text": pregunta
                }
            ]
        }
    ]

    respuesta = client.messages.create(
        model=MODELO,
        max_tokens=256,
        messages=mensajes
    )

    uso = respuesta.usage
    cache_write = getattr(uso, "cache_creation_input_tokens", 0) or 0
    cache_read = getattr(uso, "cache_read_input_tokens", 0) or 0
    tokens_normales = uso.input_tokens - cache_write - cache_read
    tokens_salida = uso.output_tokens

    coste_real = calcular_coste(tokens_normales, tokens_salida, cache_write, cache_read)
    coste_sin_cache = calcular_coste(uso.input_tokens, tokens_salida)

    resultado = {
        "llamada": i,
        "pregunta": pregunta[:50] + "...",
        "tokens_entrada_normal": tokens_normales,
        "cache_write_tokens": cache_write,
        "cache_read_tokens": cache_read,
        "tokens_salida": tokens_salida,
        "coste_real_usd": round(coste_real, 6),
        "coste_sin_cache_usd": round(coste_sin_cache, 6),
        "ahorro_usd": round(coste_sin_cache - coste_real, 6)
    }
    resultados_cache.append(resultado)

    tipo = "WRITE" if cache_write > 0 else ("READ" if cache_read > 0 else "NORMAL")
    print(f"Llamada {i} [{tipo:6s}]: "
          f"cache_write={cache_write:4d} | cache_read={cache_read:4d} | "
          f"coste=${coste_real:.5f} | sin_cache=${coste_sin_cache:.5f}")

print("-" * 80)

## 5. Calculadora de ahorro

In [ ]:
df_cache = pd.DataFrame(resultados_cache)

print("=== RESUMEN DE AHORRO CON PROMPT CACHING ===")
print(df_cache[["llamada", "cache_write_tokens", "cache_read_tokens",
               "coste_real_usd", "coste_sin_cache_usd", "ahorro_usd"]].to_string(index=False))

coste_total_real = df_cache["coste_real_usd"].sum()
coste_total_sin_cache = df_cache["coste_sin_cache_usd"].sum()
ahorro_total = df_cache["ahorro_usd"].sum()
pct_ahorro = (ahorro_total / coste_total_sin_cache) * 100

print(f"\nCoste total SIN caché: ${coste_total_sin_cache:.6f} USD")
print(f"Coste total CON caché: ${coste_total_real:.6f} USD")
print(f"Ahorro total:          ${ahorro_total:.6f} USD ({pct_ahorro:.1f}%)")

# Proyección a escala
print("\n=== PROYECCIÓN A ESCALA ===")
for n_llamadas in [100, 1_000, 10_000, 100_000]:
    # Primera llamada siempre escribe al caché; resto leen
    coste_primera = df_cache.iloc[0]["coste_real_usd"]
    coste_resto = df_cache.iloc[1]["coste_real_usd"]  # con cache_read
    coste_sin = df_cache.iloc[0]["coste_sin_cache_usd"]

    total_con = coste_primera + (n_llamadas - 1) * coste_resto
    total_sin = n_llamadas * coste_sin
    ahorro = total_sin - total_con
    print(f"  {n_llamadas:7,} llamadas: sin_caché=${total_sin:.2f} | con_caché=${total_con:.2f} | ahorro=${ahorro:.2f}")

## 6. Comparativa visual de coste con/sin caché

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Prompt Caching: Impacto en Costos", fontsize=13, fontweight="bold")

# Gráfica 1: Coste por llamada (con vs sin caché)
ax1 = axes[0]
x = df_cache["llamada"]
ancho = 0.35
bars1 = ax1.bar(x - ancho/2, df_cache["coste_sin_cache_usd"] * 1000,
                ancho, label="Sin caché", color="#e74c3c", alpha=0.8)
bars2 = ax1.bar(x + ancho/2, df_cache["coste_real_usd"] * 1000,
                ancho, label="Con caché", color="#2ecc71", alpha=0.8)
ax1.set_title("Coste por Llamada")
ax1.set_xlabel("Número de llamada")
ax1.set_ylabel("Coste (millicents USD)")
ax1.set_xticks(x)
ax1.set_xticklabels([f"Llamada {i}" for i in x])
ax1.legend()
ax1.grid(axis="y", alpha=0.3)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

# Gráfica 2: Ahorro proyectado a escala
ax2 = axes[1]
escalas = [10, 100, 1_000, 10_000, 100_000]
costes_con = []
costes_sin = []
coste_primera = df_cache.iloc[0]["coste_real_usd"]
coste_resto = df_cache.iloc[1]["coste_real_usd"]
coste_sin_unitario = df_cache.iloc[0]["coste_sin_cache_usd"]

for n in escalas:
    costes_con.append(coste_primera + (n - 1) * coste_resto)
    costes_sin.append(n * coste_sin_unitario)

ax2.plot(escalas, costes_sin, marker="o", label="Sin caché", color="#e74c3c", linewidth=2)
ax2.plot(escalas, costes_con, marker="s", label="Con caché", color="#2ecc71", linewidth=2)
ax2.fill_between(escalas, costes_sin, costes_con, alpha=0.2, color="#f39c12", label="Ahorro")
ax2.set_xscale("log")
ax2.set_title("Proyección de Ahorro a Escala")
ax2.set_xlabel("Número de llamadas (escala log)")
ax2.set_ylabel("Coste total (USD)")
ax2.legend()
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("optimizacion_costos.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Ahorro con caché en esta demo: {pct_ahorro:.1f}%")